In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

spark = SparkSession.builder.appName("AutoTec_Regresion_Completa").getOrCreate()

# Cargar datos
df = spark.read.csv("/home/jovyan/work/autotec/trabajoneiel/Semana 15/datos_automotriz_dashboard.csv", header=True, inferSchema=True)

# Variable objetivo
target = "precio"

# Variables explicativas
columnas_explicativas = [
    "kilometraje",
    "anio",
    "marca",
    "modelo",
    "combustible",
    "ciudad",
    "tipo_marca",
    "antiguedad_auto",
    "rango_kilometraje",
    "uso_anual_estimado",
    "es_ecologico"
]

# Quedarse solo con columnas existentes
columnas_explicativas = [c for c in columnas_explicativas if c in df.columns]

# Quitar nulos
df = df.dropna(subset=columnas_explicativas + [target])

# Separar categóricas y numéricas
cat_cols = [c for c in columnas_explicativas if dict(df.dtypes)[c] == "string"]
num_cols = [c for c in columnas_explicativas if c not in cat_cols]

# Indexado y one-hot para categóricas
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in cat_cols
]

encoder = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in cat_cols],
    outputCols=[f"{c}_oh" for c in cat_cols],
    handleInvalid="keep"
)

# Variables finales para entrenamiento
input_features = num_cols + [f"{c}_oh" for c in cat_cols]

assembler = VectorAssembler(
    inputCols=input_features,
    outputCol="features"
)

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaledFeatures"
)

lr = LinearRegression(featuresCol="scaledFeatures", labelCol=target)

pipeline = Pipeline(stages=indexers + [encoder, assembler, scaler, lr])

train_df, test_df = df.randomSplit([0.7, 0.3], seed=42)

model = pipeline.fit(train_df)
predictions = model.transform(test_df)

# Evaluación
evaluator_r2 = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName="r2")
evaluator_rmse = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName="rmse")
evaluator_mae = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName="mae")

r2 = evaluator_r2.evaluate(predictions)
rmse = evaluator_rmse.evaluate(predictions)
mae = evaluator_mae.evaluate(predictions)

print("=" * 60)
print("EVALUACIÓN DEL MODELO DE REGRESIÓN COMPLETA")
print("=" * 60)
print(f"R2: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print("=" * 60)

## Resultados: 
El modelo de regresión supervisada lineal presentó un desempeño inferior al Random Forest, con un R2 2
  de 0.5831 y errores MAE y RMSE más altos. Esto sugiere que la relación entre el precio y las variables explicativas no es totalmente lineal, sino que presenta interacciones complejas entre kilometraje, antigüedad, marca y modelo. Debido a ello, Random Forest logra capturar mejor la estructura del mercado automotriz y genera predicciones más precisas